# Feature QC and case-level profiles


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Imports and inputs


In [ ]:
import json
import numpy as np
import pandas as pd
from pathology.profiles import load_case_features, normalise_label
RUN = PATHS.run
DATA = RUN / 'feature_characterisation' / 'feature_statistics' / 'data'
DATA.mkdir(parents=True, exist_ok=True)


## Patch and case QC


In [ ]:
case_features, feature_columns, roi_features = load_case_features(RUN)
patch_index = pd.read_parquet(RUN / 'patch_index.parquet')
wsi = pd.read_csv(RUN / 'wsi_summary.csv')
roi = pd.read_csv(RUN / 'roi_summary.csv')
if len(feature_columns) != 768:
    raise ValueError('Expected 768 KRONOS2 dimensions')
if patch_index.patch_id.duplicated().any():
    raise ValueError('Duplicate patch identifiers')
required_coordinates = ['patch_x', 'patch_y', 'he_full_x', 'he_full_y', 'mif_full_center_x', 'mif_full_center_y']
if patch_index[required_coordinates].isna().any().any():
    raise ValueError('Missing patch coordinates')
for path in sorted((RUN / 'cases').glob('*/features.parquet')):
    matrix = pd.read_parquet(path, columns=feature_columns).to_numpy(dtype=float)
    if not np.isfinite(matrix).all():
        raise ValueError(f'Non-finite embeddings in {path.parent.name}')


## Clinical labels


In [ ]:
case_features['pcr_label'] = np.nan
if PATHS.clinical_labels.is_file():
    labels = pd.read_csv(PATHS.clinical_labels, dtype={'case_id': str})
    if not {'case_id', 'pcr_label'}.issubset(labels.columns):
        raise ValueError('Clinical CSV must contain case_id and pcr_label')
    labels = labels[['case_id', 'pcr_label']].copy()
    if labels.case_id.duplicated().any():
        raise ValueError('Duplicate clinical case identifiers')
    labels['pcr_label'] = labels.pcr_label.map(normalise_label)
    case_features = case_features.drop(columns='pcr_label').merge(labels, on='case_id', how='left', validate='one_to_one')
case_features['pcr_label'] = pd.to_numeric(case_features.pcr_label, errors='raise')


## Export analysis inputs


In [ ]:
case_qc = wsi.merge(case_features[['case_id', 'pcr_label']], on='case_id', how='left', validate='one_to_one')
case_qc['skip_fraction'] = (case_qc.skipped_patch_count / case_qc.candidate_patch_count.replace(0, np.nan)).fillna(0)
roi_qc = roi.merge(case_features[['case_id', 'pcr_label']], on='case_id', how='left', validate='many_to_one')
case_features.to_parquet(DATA / 'case_level_features_final.parquet', index=False)
roi_features.to_parquet(DATA / 'roi_level_features_final.parquet', index=False)
patch_index.to_parquet(DATA / 'patch_index_final.parquet', index=False)
case_qc.to_csv(DATA / 'final_case_qc.csv', index=False)
roi_qc.to_csv(DATA / 'final_roi_qc.csv', index=False)
summary = {'cases': len(case_features), 'labelled_cases': int(case_features.pcr_label.notna().sum()), 'selected_rois': len(roi), 'retained_rois': len(patch_index[['case_id', 'roi_id']].drop_duplicates()), 'patches': len(patch_index), 'feature_dim': len(feature_columns)}
(DATA.parent / 'qc_manifest.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.Series(summary, name='value')
